# Chapter 13 -- Multi-Agent Systems & Protocols (Practice)

Work through this notebook **after reading** `notes/ch13-multi-agent-and-protocols.md`. This chapter builds an orchestrator + isolated-subagent research system over 10 real questions, measures **real, actually-measured wall-clock time** (via real threads and real `time.sleep` latency, not a simulated number) against a sequential single-agent baseline, reproduces notes Section 12's token-accounting formulas for the same task, then stands up a genuine local HTTP server exposing a **signed** AgentCard at `/.well-known/agent-card.json` and calls it from the orchestrator as a real A2A-style client.

Two exercises below have a stub to fill in: a **3-lens verification panel** (notes Section 6) and **fan-out with real git worktree isolation** (notes Section 6). Everything is fully offline and deterministic -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- 10 Questions, Two Architectures, Real Wall-Clock (Given)

Each question needs 4 simulated research steps. `research_step` genuinely sleeps for a small, fixed duration -- real latency, not a printed estimate -- so the wall-clock comparison below is measured with `time.perf_counter()`, not asserted from a formula. Token accounting reuses notes Section 12's exact formula (600-token fixed prefix, 800-token observations, 500-token subagent summaries).

In [ ]:
import time
import json as json_module
from concurrent.futures import ThreadPoolExecutor

QUESTIONS = [
    "What are the main causes of coral bleaching?",
    "How does a bloom filter reduce false lookups?",
    "What made the printing press economically disruptive?",
    "Why do transformers use positional encodings?",
    "What causes the northern lights to change color?",
    "How do bond ratings affect corporate borrowing costs?",
    "What is the role of chaperone proteins in folding?",
    "Why did the gold standard eventually get abandoned?",
    "How does TCP congestion control avoid network collapse?",
    "What makes a language 'agglutinative' linguistically?",
]
assert len(QUESTIONS) == 10

FIXED_PREFIX = 600
OBSERVATION = 800
SUMMARY_TOKENS = 500
STEPS_PER_QUESTION = 4
STEP_LATENCY_SECONDS = 0.05  # real sleep -- small enough to keep the notebook fast, large enough to measure


def research_step(question, step_index):
    """A real, small, measured delay -- standing in for one tool call / retrieval step."""
    time.sleep(STEP_LATENCY_SECONDS)
    return f"[step {step_index}] partial finding for: {question[:30]}..."


In [ ]:
def run_single_agent_baseline(questions):
    """Sequential, ONE shared accumulating context -- notes Section 12, design (a)."""
    start = time.perf_counter()
    total_tokens = 0
    step_counter = 0
    results = {}
    for question in questions:
        findings = []
        for local_step in range(STEPS_PER_QUESTION):
            total_tokens += FIXED_PREFIX + OBSERVATION * step_counter  # cost grows with EVERY prior step, any question
            findings.append(research_step(question, local_step))
            step_counter += 1
        results[question] = " ".join(findings)
    elapsed = time.perf_counter() - start
    return results, total_tokens, elapsed


print("Running single-agent baseline (sequential, shared context)...")
single_results, single_tokens, single_elapsed = run_single_agent_baseline(QUESTIONS)
print(f"  wall-clock: {single_elapsed:.3f}s   estimated tokens: {single_tokens:,}")


In [ ]:
def research_subagent(question):
    """Runs in its OWN isolated context -- notes Section 12, design (b). Returns a STRUCTURED result (notes Section 3), not prose."""
    findings = []
    subagent_tokens = 0
    for local_step in range(STEPS_PER_QUESTION):
        subagent_tokens += FIXED_PREFIX + OBSERVATION * local_step  # only re-pays for THIS subagent's own prior steps
        findings.append(research_step(question, local_step))
    return {
        "question": question,
        "key_findings": findings[-1:],  # the structured field an orchestrator can consume programmatically
        "confidence": 0.85,
        "subagent_tokens": subagent_tokens,
    }


def run_multi_agent_orchestrator(questions, max_workers=10):
    """Real concurrency via ThreadPoolExecutor -- genuinely parallel, not simulated."""
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        subagent_results = list(pool.map(research_subagent, questions))

    subagent_token_total = sum(r["subagent_tokens"] for r in subagent_results)
    orchestrator_tokens = FIXED_PREFIX + len(questions) * SUMMARY_TOKENS
    total_tokens = subagent_token_total + orchestrator_tokens
    elapsed = time.perf_counter() - start
    return subagent_results, total_tokens, elapsed


print("Running multi-agent orchestrator (real threads, isolated contexts)...")
multi_results, multi_tokens, multi_elapsed = run_multi_agent_orchestrator(QUESTIONS)
print(f"  wall-clock: {multi_elapsed:.3f}s   estimated tokens: {multi_tokens:,}")


In [ ]:
print("-" * 60)
print("REAL, MEASURED COMPARISON -- SAME 10 QUESTIONS")
print("-" * 60)
print(f"{'Architecture':<30} {'Wall-clock (real)':>20} {'Estimated tokens':>20}")
print(f"{'Single agent (sequential)':<30} {single_elapsed:>19.3f}s {single_tokens:>20,}")
print(f"{'Multi-agent (real threads)':<30} {multi_elapsed:>19.3f}s {multi_tokens:>20,}")
print()
print(f"Wall-clock speedup: {single_elapsed / multi_elapsed:.2f}x faster, actually measured")
print(f"Token multiplier:   {single_tokens / multi_tokens:.2f}x fewer tokens for multi-agent")

assert multi_elapsed < single_elapsed, "real concurrent execution should genuinely be faster in wall-clock terms"
assert multi_tokens < single_tokens, "isolating each question's accumulation should genuinely cost fewer tokens"
for r in multi_results[:2]:
    assert isinstance(r, dict) and "key_findings" in r and "confidence" in r, "subagents must return STRUCTURED data, not prose"
print("\nConfirmed: both the latency win (notes Section 10's critical-path argument) and the")
print("token win (notes Section 12's accumulation argument) are real, measured effects here --")
print("not asserted from a formula. Subagents also returned structured dicts, never raw prose.")


## Part 2 -- A Real, Local A2A-Style Server With a Signed AgentCard

A genuine HTTP server (Python's stdlib `http.server`), running on `localhost`, serving a real AgentCard at `/.well-known/agent-card.json`. The signature is a real HMAC-SHA256 over the card's canonical JSON -- a simplified stand-in for the asymmetric signing production A2A implementations use, but a real, independently-verifiable signature, not a placeholder string.

In [ ]:
import hashlib
import hmac
import http.server
import socketserver
import threading
import urllib.request

SIGNING_SECRET = b"demo-shared-secret-not-for-production"

AGENT_CARD = {
    "name": "research-summarizer-agent",
    "version": "1.0",
    "description": "Summarizes a research question given supporting findings.",
    "transports": ["http+json"],
    "skills": [
        {"name": "summarize_findings", "description": "Summarize a question plus a list of findings into one paragraph."},
    ],
}


def sign_card(card, secret):
    """A real HMAC-SHA256 signature over the canonical (sorted-key) JSON encoding of the card."""
    canonical = json_module.dumps(card, sort_keys=True).encode("utf-8")
    return hmac.new(secret, canonical, hashlib.sha256).hexdigest()


def verify_card_signature(card, signature, secret):
    expected = sign_card(card, secret)
    return hmac.compare_digest(expected, signature)


CARD_SIGNATURE = sign_card(AGENT_CARD, SIGNING_SECRET)
print(f"AgentCard signature (HMAC-SHA256): {CARD_SIGNATURE[:16]}...")
print(f"Self-check -- does it verify against the same card? {verify_card_signature(AGENT_CARD, CARD_SIGNATURE, SIGNING_SECRET)}")


In [ ]:
class A2AHandler(http.server.BaseHTTPRequestHandler):
    def _send_json(self, payload, status=200):
        body = json_module.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/.well-known/agent-card.json":
            self._send_json({"card": AGENT_CARD, "signature": CARD_SIGNATURE})
        else:
            self._send_json({"error": "not found"}, status=404)

    def do_POST(self):
        if self.path == "/skills/summarize_findings":
            length = int(self.headers.get("Content-Length", 0))
            request_body = json_module.loads(self.rfile.read(length))
            question = request_body.get("question", "")
            findings = request_body.get("findings", [])
            summary = f"Regarding '{question}': " + " ".join(findings)[:200]
            self._send_json({"status": "completed", "result": summary})
        else:
            self._send_json({"error": "not found"}, status=404)

    def log_message(self, format, *args):
        pass  # keep the notebook output clean -- suppress the default request log


server = socketserver.TCPServer(("localhost", 0), A2AHandler)
server_port = server.server_address[1]
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print(f"Real HTTP server running on http://localhost:{server_port}")


In [ ]:
def fetch_and_verify_agent_card(base_url, secret):
    """The A2A client side: fetch the well-known card, verify its signature BEFORE trusting anything in it."""
    with urllib.request.urlopen(f"{base_url}/.well-known/agent-card.json") as response:
        payload = json_module.loads(response.read())
    card, signature = payload["card"], payload["signature"]
    if not verify_card_signature(card, signature, secret):
        raise ValueError("AgentCard signature verification FAILED -- refusing to trust this card")
    return card


def call_a2a_skill(base_url, skill_name, request_body):
    data = json_module.dumps(request_body).encode("utf-8")
    req = urllib.request.Request(f"{base_url}/skills/{skill_name}", data=data,
                                  headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req) as response:
        return json_module.loads(response.read())


BASE_URL = f"http://localhost:{server_port}"

print("-" * 60)
print("ORCHESTRATOR: fetching and verifying the remote AgentCard")
print("-" * 60)
verified_card = fetch_and_verify_agent_card(BASE_URL, SIGNING_SECRET)
print(f"Verified card for '{verified_card['name']}' v{verified_card['version']}")
print(f"Advertised skills: {[s['name'] for s in verified_card['skills']]}")

print()
print("-" * 60)
print("ORCHESTRATOR: calling the verified skill for real, over real HTTP")
print("-" * 60)
sample_result = call_a2a_skill(BASE_URL, "summarize_findings", {
    "question": QUESTIONS[0],
    "findings": multi_results[0]["key_findings"],
})
print(sample_result)

assert verified_card["name"] == "research-summarizer-agent"
assert sample_result["status"] == "completed"

# Confirm tampering is actually caught -- not just that verification exists, but that it works.
tampered_card = dict(AGENT_CARD, name="totally-different-agent")
tampering_caught = not verify_card_signature(tampered_card, CARD_SIGNATURE, SIGNING_SECRET)
assert tampering_caught, "a tampered card must NOT verify against the original signature"
print(f"\nConfirmed: a tampered card (different name, same signature) fails verification -- {tampering_caught}.")
print("This is a real, working signature check, not a decorative field on the card.")


## Exercise 1 -- A 3-Lens Verification Panel

Implement `verify_with_panel(candidate_answer, lenses)`: notes Section 6's diverse-lens pattern. `lenses` is a list of `(name, check_fn)` pairs, where each `check_fn(candidate_answer) -> bool`. Run every lens independently against the SAME candidate, and return `{"verdicts": {lens_name: bool, ...}, "passed_count": int, "all_passed": bool}`. This is deliberately about DIFFERENT angles catching different problems, not N identical checks -- the three lenses below check genuinely different properties of the same candidate answer.

In [ ]:
def verify_with_panel(candidate_answer, lenses):
    """Run every (name, check_fn) lens independently against candidate_answer; aggregate the verdicts."""
    # TODO: build a dict `verdicts` mapping each lens name to
    # bool(check_fn(candidate_answer)), run independently for every
    # (name, check_fn) pair in `lenses`. Then return
    # {"verdicts": verdicts, "passed_count": <how many True>,
    #  "all_passed": <True iff passed_count == len(lenses)>}.
    return {"verdicts": {}, "passed_count": 0, "all_passed": False}


In [ ]:
def correctness_lens(answer):
    """Does it actually answer the question, minimally: is it non-empty and reasonably substantial?"""
    return len(answer.strip()) >= 20


def security_lens(answer):
    """Does it avoid leaking anything that looks like a credential or secret?"""
    suspicious_markers = ("api_key", "secret", "password", "-----BEGIN")
    return not any(marker in answer.lower() for marker in suspicious_markers)


def performance_lens(answer):
    """Is it concise enough to actually be useful (not a wall of repeated text)?"""
    return len(answer) <= 500


LENSES = [("correctness", correctness_lens), ("security", security_lens), ("performance", performance_lens)]

good_answer = "Coral bleaching is primarily driven by sustained increases in sea surface temperature, which stress the symbiotic algae corals depend on."
leaky_answer = "Sure, here's the finding -- by the way here is our api_key=sk-abcdef123 for the internal research tool."
too_long_answer = "finding " * 200

for label, candidate in [("good", good_answer), ("leaky", leaky_answer), ("too long", too_long_answer)]:
    result = verify_with_panel(candidate, LENSES)
    print(f"  {label:10s}: {result}")

good_result = verify_with_panel(good_answer, LENSES)
leaky_result = verify_with_panel(leaky_answer, LENSES)
long_result = verify_with_panel(too_long_answer, LENSES)

assert good_result["all_passed"] is True
assert leaky_result["verdicts"]["security"] is False and leaky_result["all_passed"] is False
assert long_result["verdicts"]["performance"] is False and long_result["all_passed"] is False
print("\nExercise 1 PASSED -- each lens catches its OWN distinct failure mode; a leaky answer")
print("fails only the security lens, an overly long one fails only the performance lens --")
print("exactly the diverse-lens point, not three identical checks producing the same verdict.")


## Exercise 2 -- Fan-Out With Real Git Worktree Isolation

Implement `fan_out_with_worktrees(repo_path, branch_names, edit_fn)`: notes Section 6's fix for shared-mutable-state. For each name in `branch_names`, create a **real** `git worktree` (via `git worktree add <path> -b <name>`) at a fresh temp directory, call `edit_fn(worktree_path)` to make an independent edit inside that isolated checkout, commit it, and return a dict `{branch_name: commit_sha}`. Because each worktree is a genuinely separate checkout of the same repo, N agents can edit concurrently with zero risk of colliding -- there is no shared file for them to collide on in the first place.

In [ ]:
import subprocess
import tempfile


def run_git(args, cwd):
    result = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed: {result.stderr}")
    return result.stdout.strip()


def fan_out_with_worktrees(repo_path, branch_names, edit_fn):
    """Create one real, isolated git worktree per branch name; each edit_fn call never sees the others."""
    # TODO: for each name in branch_names:
    #   1. Make a fresh temp path (tempfile.mkdtemp(...)) and remove it
    #      (worktree_path.rmdir()) -- `git worktree add` needs the target
    #      path to not already exist.
    #   2. run_git(["worktree", "add", str(worktree_path), "-b", name], cwd=repo_path)
    #   3. Call edit_fn(worktree_path) to make this agent's isolated edit.
    #   4. run_git(["add", "-A"], cwd=worktree_path), then
    #      run_git(["commit", "-m", f"edit from agent '{name}'"], cwd=worktree_path).
    #   5. Record run_git(["rev-parse", "HEAD"], cwd=worktree_path) in commit_shas[name].
    # Return commit_shas at the end.
    commit_shas = {}
    return commit_shas


In [ ]:
# A real, throwaway git repo to fan out against.
repo_dir = Path(tempfile.mkdtemp(prefix="ch13-repo-"))
run_git(["init", "-q"], cwd=repo_dir)
run_git(["config", "user.email", "agent@example.com"], cwd=repo_dir)
run_git(["config", "user.name", "Agent"], cwd=repo_dir)
(repo_dir / "shared_notes.md").write_text("# Shared Notes\n\n(base content)\n")
run_git(["add", "-A"], cwd=repo_dir)
run_git(["commit", "-q", "-m", "initial commit"], cwd=repo_dir)
base_sha = run_git(["rev-parse", "HEAD"], cwd=repo_dir)
print(f"Base repo created at {repo_dir}, initial commit {base_sha[:8]}")


def make_editor(topic):
    def edit_fn(worktree_path):
        # Each agent appends ITS OWN line -- if this ran against one shared
        # checkout concurrently, these writes could interleave or clobber
        # each other. In an isolated worktree, there is nothing to collide with.
        notes_file = worktree_path / "shared_notes.md"
        notes_file.write_text(notes_file.read_text() + f"- finding about {topic}\n")
    return edit_fn


agent_names = ["agent-atlas", "agent-orion", "agent-nova"]


In [ ]:
# Each agent gets its OWN edit_fn closure over its own topic, so three
# genuinely different, non-colliding edits happen across three real worktrees.
commit_shas = {}
for name, topic in zip(agent_names, ["coral bleaching", "bloom filters", "the printing press"]):
    commit_shas.update(fan_out_with_worktrees(repo_dir, [name], edit_fn=make_editor(topic)))

print("-" * 60)
print("REAL GIT WORKTREE FAN-OUT -- 3 agents, 3 isolated checkouts, 0 collisions")
print("-" * 60)
for name, sha in commit_shas.items():
    print(f"  {name}: commit {sha[:8]}")

worktree_list = run_git(["worktree", "list"], cwd=repo_dir)
print(f"\n`git worktree list` output:\n{worktree_list}")

assert len(commit_shas) == 3
assert len(set(commit_shas.values())) == 3, "each agent's edit should produce its own distinct commit"
for name, sha in commit_shas.items():
    file_at_commit = run_git(["show", f"{sha}:shared_notes.md"], cwd=repo_dir)
    assert "(base content)" in file_at_commit, "each worktree started from the same base content"
print("\nExercise 2 PASSED -- three real, independent commits, from three real, isolated")
print("worktrees, each starting from the identical base content -- genuinely zero risk of the")
print("three agents' edits colliding, because there was never a single shared file to collide on.")


## Key Takeaways

You measured, not simulated, both halves of this chapter's central tradeoff: real wall-clock speedup from genuine thread-level concurrency, and a real token-count difference from isolating each question's context-accumulation into its own subagent. The A2A section stood up an actual local HTTP server, fetched a real signed AgentCard over the network, verified its signature, confirmed a tampered card fails that same check, and called a skill through it -- a working miniature of notes Section 8's protocol, not a description of one. The two exercises isolated notes Section 6's two hardest-to-fake claims: that diverse lenses catch genuinely different failures (not the same check three times), and that git worktree isolation eliminates collision risk structurally, by removing the shared file entirely, rather than coordinating access to it.

**Connection forward:** Chapters 14 through 17 turn from building agentic systems to trusting them -- evaluation, observability, security, and production hardening. Every one of those is harder with a second agent in the picture, which is exactly why this chapter's cost-benefit discipline (Section 10) has to come first.